# AI / ML: Deep Learning – GenAI - Transformers
- Text Summarizer App (HuggingFace Transformer)

In [40]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
import re
import torch

## Reading Data:

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [5]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data.sample(10)

,id,dialogue,summary
13049,13818905,Monica: pls come with me <file_other>\r\nMelin...,Monica doesn't want to come alone to see the w...
8417,13829098,Kate: hi! Have you seen the apartment?\r\nKare...,Karen will get mortgage to buy an apartment. K...
7554,13828837,"Hugh: Hi, have you decided whether you're goin...",Grant will write an essay on topic 2 or 3. He ...
167,13680415,Sam: can u turn down the music? I'm trying to ...,Fan will turn down the music at Sam's request.
7516,13820574,Amber: <file_photo>\r\nSophia: O wow!!\r\nIsla...,Amber looked stunning during her prom. Sophia ...
2593,13862363,"Alvin: hey, what's up? how are the works going...",Margot found out from Chris that there will be...
10891,13829793,Peter: how the weather in Barcelona? \r\nPeter...,Peter is experiencing heavy snowfall and sends...
134,13862751,Lorry: I'm running 10 min late..\nTereza: Ok d...,Tereza is waiting with Tony at the main car pa...
1118,13730265,Andy: hey Ali :)) \r\nAlison: hey Andy :)) all...,Andy is dressing up as the Incredible Hulk for...
11966,13812182,Lilah: Teacher is asking me to visit his home\...,Lilah will Pick up Pedro from his house in 10 ...


In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

## Random Sampeling:

In [9]:
train_data = train_data.sample(
    n = 4000,
    random_state = 42
).reset_index(drop = True)

val_data = val_data.sample(
    n = 500,
    random_state = 42
).reset_index(drop = True)

In [10]:
train_data.shape

(4000, 3)

In [11]:
val_data.shape

(500, 3)

## Data Preprocessing: Cleaning Data

In [12]:
def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"\<.*?>", " ", text) # html tags
    text = text.strip().lower()
    return text

In [13]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [14]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

## Tokenization our Data for T5: Model

In [16]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [25]:
# Function to convert row data to tokens:
# Tokenise input for fine-tuning of our Model training.

def tokenize(data):
    inputs = tokenizer( # Input Values
        data["dialogue"],
        padding = "max_length",
        max_length = 512,
        truncation = True
    )

    targets = tokenizer( # Output Values
        data["summary"],
        padding = "max_length",
        max_length = 150,
        truncation = True
    )

    inputs["labels"] = targets["input_ids"] # Token ids => added to input as labels

    return inputs

## Calling the 'tokenize()' function
- For Training & validation Data.

In [26]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [35]:
train_dataset[0]

# input ids - dialogues (token ids) 
# attention mask - inputs id => valid values or padding vals
# labels - target vals (summary)

# 1 - end of sequences
# 0 - added paddings

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [34]:
len(train_dataset[0]["input_ids"])

512

In [36]:
type(train_dataset)

list

In [37]:
type(val_dataset)

list

## Fine-Tuning our Transformer Model:
- Working with our Model.

In [39]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [46]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

Device: cpu


In [47]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [48]:
# Training Arguments:

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

## Training our Model:

In [49]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

## Model Training: Slowest Part of the Project

In [50]:
trainer.train()

C:\Users\satin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 